# The discopt Optimization Course (`discopt tutor`)

discopt ships with a full, self-paced course in mathematical optimization, taught *through* the solver. It is a structured **30-lesson curriculum** organized into three tracks — **basic**, **intermediate**, and **advanced** — that takes you from "what is a linear program?" all the way to spatial branch-and-bound, neural-net-embedded MINLP, and differentiable optimization.

The curriculum is interactive: it is delivered through [Claude Code](https://claude.com/claude-code), which acts as your teaching assistant. It walks you through each reading, answers questions, gives graduated hints when you are stuck, and grades your exercises and writing against a rubric. The whole thing is driven by a small command-line front end, `discopt tutor`, plus a set of `/course:` slash commands.

The material aligns with the standard graduate optimization literature. The convex-analysis and duality lessons follow {cite:p}`Boyd2004`, and the nonlinear-optimization lessons (unconstrained descent, KKT conditions, interior-point methods) follow {cite:p}`Nocedal2006`. Roughly speaking, the full course is equivalent to a one-year graduate sequence in optimization.

By the end of this notebook you will know:

1. What the course is and how it is structured.
2. How to drive it from the `discopt tutor` CLI.
3. What each of the 30 lessons covers.
4. Where to go for the non-interactive, self-contained tutorials in this book.

In [1]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

## 1. The `discopt tutor` command line

`discopt tutor` is a thin CLI wrapper. It locates the packaged `course/` tree, resolves a lesson id, and launches a `claude` session running the matching `/course:` slash command. It keeps no separate state of its own — the lesson library, grading rubrics, and your progress all live under `course/`.

The cell below prints its help text. This is safe to execute: it only shells out to `discopt` (which is installed) and never to `claude`.

In [2]:
import subprocess

result = subprocess.run(
    ["discopt", "tutor", "--help"],
    capture_output=True,
    text=True,
)
print(result.stdout)

usage: discopt tutor [-h] <subcommand> ...

Entry point for the discopt course. Locates the course/ tree, resolves a
lesson id, and launches a claude session running the matching /course: slash
command.

positional arguments:
  <subcommand>
    list        list every lesson with status
    start       launch a specific lesson
    resume      continue current_lesson
    next        start the next-numbered lesson
    reset       reset progress (one lesson, or all)
    install     install /course: slash commands into ./.claude/

options:
  -h, --help    show this help message and exit



The subcommands break down as follows:

| Command | What it does |
| ------- | ------------ |
| `discopt tutor` | Dashboard: per-track completion counts, average scores, and the next lesson. |
| `discopt tutor list` | List every lesson with its completion status. |
| `discopt tutor start <lesson>` | Launch `claude /course:lesson <lesson>` for a specific lesson. |
| `discopt tutor resume` | Start whatever `current_lesson` is in your progress file. |
| `discopt tutor next` | Start the next-numbered lesson after `current_lesson`. |
| `discopt tutor reset [<lesson>]` | Drop one progress entry (or reset everything). |
| `discopt tutor install [--force]` | Materialize a writable `course/` copy and install the `/course:` slash commands into `./.claude/`. |

A lesson id can be given canonically (`basic/02_lp_fundamentals`), by short name (`02_lp_fundamentals`), or as a bare number (`02`); the CLI resolves it across tracks.

### Where the course content lives

The course ships as package data under `discopt.course`, so a plain `pip install discopt` includes it. The CLI finds the tree in this order:

1. `$DISCOPT_COURSE_DIR`, if set.
2. A writable `course/SYLLABUS.md` found by walking up from the current directory (so a copy materialized into your own project always wins).
3. The read-only packaged copy shipped inside the wheel.

Running `discopt tutor install` copies the packaged tree into `./course/` and drops the `/course:` slash commands and the `course-assessor` skill into `./.claude/`, giving you a writable working copy.

Below we read the syllabus directly from the packaged course data. This executes cleanly — it is just file I/O against the installed package.

In [3]:
from importlib import resources

syllabus = resources.files("discopt.course").joinpath("SYLLABUS.md").read_text()
print(syllabus)

# Syllabus

Thirty lessons in three tracks. Each row links the lesson directory and the
primary literature backing it (full BibTeX entries in `docs/references.bib`).

## Basic track — modeling and solving

| #  | Title                              | Primary references                        |
| -- | ---------------------------------- | ----------------------------------------- |
| 01 | Introduction to optimization       | Bertsimas1997, Williams2013, Wolsey1998   |
| 02 | Linear programming fundamentals    | Bertsimas1997, Dantzig1963                |
| 03 | Modeling in `discopt`              | Williams2013                              |
| 04 | Sensitivity, duality, shadow prices| Bertsimas1997, Boyd2004                   |
| 05 | Integer programming basics         | Wolsey1998, Williams2013, Conforti2014    |
| 06 | Branch-and-bound, conceptually     | Land1960, Dakin1965, Wolsey1998           |
| 07 | Convexity and why it matters       | Boyd2004, Rockafellar1970                 |
| 

## 2. The learning workflow

Each lesson lives in `course/<track>/<id>/` and pairs four files:

| File | Purpose |
| ---- | ------- |
| `reading.ipynb` | The lesson itself — math, examples, and runnable `discopt` code. |
| `exercises.ipynb` | 3–6 exercises with `# TODO` cells you fill in. |
| `writing.md` | A short essay / analysis prompt (about one page). |
| `rubric.md` | The criteria Claude uses to assess your work. |

Claude Code is the teaching assistant. The `discopt tutor` subcommands launch a `claude` session running one of these `/course:` slash commands:

- `/course:lesson <id>` — load a lesson and get a guided tour of the reading.
- `/course:hint <id> <ex#>` — get a graduated hint for a specific exercise (without revealing the answer).
- `/course:assess <id>` — Claude grades your exercises against the rubric.
- `/course:grade-writing <id>` — Claude grades your `writing.md` response.
- `/course:progress` — show what you have completed and what is next.
- `/course:cite-check` — verify all `{cite:p}` keys resolve in `references.bib`.

The grading behavior is implemented by the `course-assessor` skill, so it stays consistent across sessions.

### A typical session

The commands below shell out to the `claude` CLI to open an interactive tutoring session, so they are **not executed here** — they are shown for reference. Run them in your own terminal (with Claude Code installed) inside a `discopt` checkout or a directory where you have run `discopt tutor install`.

```bash
# One-time: materialize a writable course/ and install the /course: commands
discopt tutor install

# See where you stand
discopt tutor              # dashboard: counts, averages, next lesson
discopt tutor list         # every lesson with its score / status

# Start your first lesson (any of these forms resolves the same lesson)
discopt tutor start basic/01_intro_to_optimization
discopt tutor start 01_intro_to_optimization
discopt tutor start 01

# Pick up where you left off, or move on to the next lesson
discopt tutor resume
discopt tutor next

# Reset progress for one lesson, or all of it
discopt tutor reset basic/01_intro_to_optimization
discopt tutor reset
```

Inside the launched `claude` session you interact with the `/course:` slash commands directly:

```text
/course:lesson basic/01_intro_to_optimization
/course:hint basic/01_intro_to_optimization 2
/course:assess basic/01_intro_to_optimization
/course:grade-writing basic/01_intro_to_optimization
/course:progress
```

### How progress is tracked

Your state lives in `course/progress.yaml`, created from `course/progress.template.yaml` on your first `/course:progress` call. It records `current_lesson` and a `scores` map of per-lesson totals.

Each lesson awards up to 100 points spread across exercise correctness, writing quality, and (on capstones) experimental rigor. A lesson counts as **passed at ≥ 70 / 100**. `discopt tutor` reads this file directly to power the dashboard, the `list` view, and `next`/`resume`. Lessons unlock in order within a track, but the three tracks can be worked in parallel.

## 3. The curriculum at a glance

Thirty lessons across three tracks. The basic track gives a working knowledge of optimization and the `discopt` API; the intermediate track is algorithms-first; the advanced track covers research frontiers and ends in a paper-style capstone.

### Basic track (1–10) — modeling and solving

| #  | Lesson | Theme |
| -- | ------ | ----- |
| 01 | Introduction to optimization | What optimization is; problem taxonomy. |
| 02 | Linear programming fundamentals | LP standard form, geometry of the feasible region. |
| 03 | Modeling in `discopt` | The modeling API; variables, constraints, objectives. |
| 04 | Sensitivity, duality, shadow prices | LP duality and what dual values mean {cite:p}`Boyd2004`. |
| 05 | Integer programming basics | Binary / integer variables, formulation tricks. |
| 06 | Branch-and-bound, conceptually | How B&B searches the integer tree. |
| 07 | Convexity and why it matters | Convex sets and functions; why convexity is decisive {cite:p}`Boyd2004`. |
| 08 | Unconstrained nonlinear optimization | Gradients, descent, line search {cite:p}`Nocedal2006`. |
| 09 | Constrained NLP — KKT and IPOPT | KKT conditions and interior-point NLP {cite:p}`Nocedal2006`. |
| 10 | Capstone (basic) | Build and analyze a real model end to end. |

### Intermediate track (11–20) — algorithms and structure

| #  | Lesson | Theme |
| -- | ------ | ----- |
| 11 | Simplex revealed | The simplex method, pivoting, anti-cycling. |
| 12 | Interior-point methods | Barrier methods and IPMs {cite:p}`Nocedal2006`. |
| 13 | Cutting planes | Gomory and lift-and-project cuts. |
| 14 | Branching rules | Variable selection and pseudocost branching. |
| 15 | Presolve and FBBT | Bound tightening and feasibility-based propagation. |
| 16 | Convex relaxations | McCormick and other relaxation schemes. |
| 17 | Conic optimization (SOCP/SDP) | Second-order-cone and semidefinite programs {cite:p}`Boyd2004`. |
| 18 | Decomposition methods | Benders and Dantzig–Wolfe decomposition. |
| 19 | Stochastic programming | Recourse models and scenario formulations. |
| 20 | Capstone (intermediate) | Implement a piece of an algorithm and benchmark it. |

### Advanced track (21–30) — frontiers

| #  | Lesson | Theme |
| -- | ------ | ----- |
| 21 | Spatial branch-and-bound | Global B&B over continuous nonconvexities. |
| 22 | Global optimization theory | Convergence and convex underestimators. |
| 23 | Generalized disjunctive programming | Disjunctions, hull vs. big-M reformulations. |
| 24 | Neural-net-embedded MINLP | Embedding trained networks as constraints. |
| 25 | Robust optimization | Uncertainty sets and robust counterparts. |
| 26 | Differentiable optimization | Differentiating through optimization layers. |
| 27 | Machine learning for optimization | Learned heuristics and branching policies. |
| 28 | Bilevel and complementarity | Bilevel programs and MPECs. |
| 29 | Reformulations and symmetry | Symmetry breaking and reformulation. |
| 30 | Capstone (advanced) | A paper-style writeup with computational results. |

The authoritative reference list per lesson is in `course/SYLLABUS.md` (printed above), with full BibTeX in `docs/references.bib`.

## 4. The non-interactive counterpart

The `discopt tutor` course is the *guided, graded* path. If you would rather read self-contained, fully-executed material on a specific topic, the rest of this book covers much of the same ground without needing Claude Code:

- [**Quickstart**](quickstart.ipynb) — your first model, mirroring lessons 1–3.
- [**Modeling guide**](modeling_guide.ipynb) — the modeling API in depth (lesson 3).
- [**LP**](tutorial_lp.ipynb), [**QP**](tutorial_qp.ipynb), [**MILP**](tutorial_milp.ipynb), [**MIQP**](tutorial_miqp.ipynb), [**MINLP**](tutorial_minlp.ipynb) — one notebook per problem class (lessons 2, 5, 8).
- [**Sensitivity analysis**](sensitivity_analysis.ipynb) — duality and shadow prices (lesson 4).
- [**Solver internals**](solver_internals.ipynb), [**Presolve**](presolve.ipynb), [**Bound tightening**](bound_tightening.ipynb) — the machinery behind lessons 6, 11–15.
- [**GDP**](tutorial_gdp.ipynb), [**Robust optimization**](tutorial_robust.ipynb), [**NN embedding**](nn_embedding.ipynb), [**Differentiable MILP**](differentiable_milp.ipynb), [**Complementarity / MPEC**](complementarity_mpec.ipynb) — the advanced-track topics (lessons 23–28).

Use these to preview a topic, then run the matching lesson with `discopt tutor start <lesson>` when you want the guided, graded version. Happy solving.